In [ ]:
import torch
import numpy as np
import nibabel as nib
from pathlib import Path
from omegaconf import OmegaConf

from src.python.functions.networks.nets import DiffusionModelUNet, VQVAE
from src.python.functions.networks.schedulers import DDPMScheduler, DDIMScheduler

# ============================================================
# ===================== USER INPUT ===========================
# ============================================================

DEVICE = "cuda"

STAGE1_CKPT = "/path/to/vqgan_checkpoint.pth"
STAGE1_CFG  = "/path/to/vqgan_config.yaml"

DIFF_CKPT   = "/path/to/diffusion_checkpoint.pth"
DIFF_CFG    = "/path/to/ldm_config.yaml"

OUT_DIR = Path("samples")
OUT_DIR.mkdir(exist_ok=True)

N_SAMPLES  = 4
TIMESTEPS  = 300
SCHEDULER  = "ddpm"   # "ddpm" or "ddim"

# dataset geometry
VOLUME_SHAPE = (512, 512, 256)
DOWNSAMPLE = 4
LATENT_SHAPE = (VOLUME_SHAPE[0]//DOWNSAMPLE,
                VOLUME_SHAPE[1]//DOWNSAMPLE,
                VOLUME_SHAPE[2]//DOWNSAMPLE)

# ============================================================
# ===================== LOAD STAGE-1 =========================
# ============================================================

class VQWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def decode(self, z):
        return self.model.decode_stage_2_outputs(z)

print("Loading VQGAN...")
stage1_cfg = OmegaConf.load(STAGE1_CFG)
stage1 = VQVAE(**stage1_cfg["stage1"]["params"])
stage1.load_state_dict(torch.load(STAGE1_CKPT, map_location="cpu")["state_dict"])
stage1 = VQWrapper(stage1).to(DEVICE).eval()

# ============================================================
# ===================== LOAD DIFFUSION =======================
# ============================================================

print("Loading diffusion model...")
diff_cfg = OmegaConf.load(DIFF_CFG)

unet = DiffusionModelUNet(**diff_cfg["ldm"]["params"])
unet.load_state_dict(torch.load(DIFF_CKPT, map_location="cpu"))
unet = unet.to(DEVICE).eval()

# Scheduler
if SCHEDULER == "ddpm":
    scheduler = DDPMScheduler(**diff_cfg["ldm"]["scheduler"])
elif SCHEDULER == "ddim":
    scheduler = DDIMScheduler(**diff_cfg["ldm"]["scheduler"])
else:
    raise ValueError("Scheduler must be ddpm or ddim")

scheduler.set_timesteps(TIMESTEPS)

# ============================================================
# ===================== SAMPLING =============================
# ============================================================

print("Sampling...")

C = diff_cfg["ldm"]["params"]["in_channels"]

for i in range(N_SAMPLES):

    x = torch.randn((1, C, *LATENT_SHAPE), device=DEVICE)

    with torch.no_grad():
        for t in scheduler.timesteps:
            eps = unet(x, timesteps=torch.tensor([t], device=DEVICE))
            x, _ = scheduler.step(eps, t, x)

        recon = stage1.decode_stage_2_outputs(x)

    # Convert to HU
    vol = np.clip(recon[0, 0].cpu().numpy(), -1, 1)
    hu = (((vol + 1) * (2000 / 2)) - 1000).astype(np.int16)

    nib.save(nib.Nifti1Image(hu, None), OUT_DIR / f"sample_{i}.nii.gz")
    print(f"Saved sample_{i}.nii.gz")

print("Done.")
